In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch
from torch.optim import AdamW

In [3]:
df = pd.read_csv('/content/drive/MyDrive/ML/cleaned_data.csv')
print(df.shape)
print(df.head())

(63675, 2)
   label                                           combined
0      1  LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1      1     Did they post their votes for Hillary already?
2      1  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3      0  Bobby Jindal, raised Hindu, uses story of Chri...
4      1  SATAN 2: Russia unvelis an image of its terrif...


In [4]:
train_df, temp_df  = train_test_split(df, test_size=0.3, random_state=42)

In [5]:
val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42)

In [6]:
print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 44572
Validation: 9551
Test: 9552


In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
sample = "This news is completely fake"
tokens = tokenizer(sample, max_length=512, truncation=True, padding='max_length', return_tensors='pt')
print(tokens)
print(tokens['input_ids'].shape)

{'input_ids': tensor([[ 101, 2023, 2739, 2003, 3294, 8275,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,

In [9]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
      text = self.texts.iloc[idx]
      encoding = self.tokenizer(
      text,
      max_length=self.max_length,
      truncation=True,
      padding='max_length',
      return_tensors='pt'
      )
      input_ids= encoding['input_ids'].squeeze()
      attention_mask=encoding['attention_mask'].squeeze()
      label=self.labels.iloc[idx]
      return input_ids, attention_mask, label

In [10]:
train_dataset = FakeNewsDataset(
    texts=train_df['combined'],
    labels=train_df['label'],
    tokenizer=tokenizer
)
test_dataset = FakeNewsDataset(
    texts=test_df['combined'],
    labels=test_df['label'],
    tokenizer=tokenizer
)
val_dataset = FakeNewsDataset(
    texts=val_df['combined'],
    labels=val_df['label'],
    tokenizer=tokenizer
)

In [11]:
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 44572
Val: 9551
Test: 9552


In [12]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [13]:
batch = next(iter(train_loader))
input_ids, attention_mask, labels = batch
print(f"input_ids shape: {input_ids.shape}")
print(f"attention_mask shape: {attention_mask.shape}")
print(f"labels shape: {labels.shape}")

input_ids shape: torch.Size([32, 512])
attention_mask shape: torch.Size([32, 512])
labels shape: torch.Size([32])


In [14]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = model.to(device)

cuda


In [16]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [17]:
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids, attention_mask, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader)}")

Epoch 1 Loss: 0.0518895668999796
Epoch 2 Loss: 0.013063163407717224
Epoch 3 Loss: 0.007063823621456616


In [18]:
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, '/content/drive/MyDrive/ML/trained_model.pth')

print("Model saved to Google Drive!")

Model saved to Google Drive!
